# This notebook is used to test and execute the label generation
The first cell is used to test it on specific examples. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import LinearSegmentedColormap

# Create a custom colormap that matches the depth map colors
colors = ['darkblue', 'purple', 'orange', 'yellow']
n_bins = 256
custom_cmap = LinearSegmentedColormap.from_list('custom', colors, N=n_bins)

def calculate_danger(depth_column, trim_ratio=0.2, sensitivity=2.0, floor_sensitivity=0.5):
    """
    Calculate a danger value (0-1) for a depth column
    
    Parameters:
    - depth_column: Numpy array of depth values (0-1 where HIGHER is CLOSER)
    - trim_ratio: Ratio to trim from top and bottom of column
    - sensitivity: How sensitive the danger score is to close objects
    - floor_sensitivity: Reduced sensitivity for floor detection (lower depth area)
    
    Returns:
    - danger_value: Value between 0-1 where 1 is dangerous
    - trim_indices: Tuple of (start_idx, end_idx) showing which part was used
    """
    # print(f"Depth column shape: {np.asarray(depth_column).shape[0]}")
    # Trim top and bottom
    n = np.asarray(depth_column).shape[0]
    trim_size = int(n * trim_ratio)
    # Ensure depth_column is a 1D array
    depth_column = np.asarray(depth_column).flatten()
    
    # Calculate trim indices
    start_idx = trim_size
    end_idx = n - trim_size
    
    # Ensure we have enough data to work with
    if n <= 2 * trim_size:
        return 0.0, (0, n)
    
    trimmed = depth_column[start_idx:end_idx]
    
    # Identify the floor (typically lower part of image has higher depth values)
    mid_point = len(trimmed) // 2
    upper_half = trimmed[:mid_point]
    lower_half = trimmed[mid_point:]
    
    # Calculate separate danger values for upper and lower portions
    # For upper portion - use full sensitivity (potential obstacles)
    if len(upper_half) > 0:
        # Use max value to detect closest objects (since higher values = closer)
        upper_depth = np.percentile(upper_half, 80)  # 80th percentile to avoid noise
        # Apply sensitivity factor (closer/higher = more dangerous)
        upper_danger = np.clip(sensitivity * upper_depth, 0, 1)
    else:
        upper_danger = 0.0
    
    # For lower portion - use reduced sensitivity (floor detection)
    if len(lower_half) > 0:
        lower_depth = np.percentile(lower_half, 80)
        # Apply reduced sensitivity for floor
        lower_danger = np.clip(floor_sensitivity * lower_depth, 0, 1)
    else:
        lower_danger = 0.0
    
    # Take the maximum danger from upper and lower portions
    danger_value = max(upper_danger, lower_danger)
    
    return danger_value, (start_idx, end_idx)

def plot_simple_depth_with_danger(image_path, depth_path, num_columns=5, depth_range=(0, 1)):
    """
    Analyze image and depth map with danger metrics across columns
    
    Parameters:
    - image_path: Path to image
    - depth_path: Path to depth map
    - num_columns: Number of columns to analyze
    - depth_range: Tuple indicating the actual range of depth values (min, max)
                   where HIGHER values mean CLOSER objects
    """
    # Load images
    img = plt.imread(image_path)
    depth = plt.imread(depth_path)
    
    # Convert depth to numpy array if needed
    if depth.dtype == np.uint8:
        depth = depth.astype(np.float32) / 255.0

    # Get actual min and max values from the depth data
    depth_min = np.min(depth)
    depth_max = np.max(depth)
    
    # Create figure with subplots - 3 plots in a column
    fig, axes = plt.subplots(3, 1, figsize=(8, 8))
    
    # Original image
    axes[0].imshow(img)
    h, w = img.shape[:2]
    
    # Center box (full height, centered width)
    box_width = int(h)  # Match image height
    box_height = int(h)
    y_start = (h - box_height) // 2
    x_start = (w - box_width) // 2
    
    # Draw center box on original image
    rect1 = plt.Rectangle((x_start, y_start), box_width, box_height,
                         linewidth=2, edgecolor='r', facecolor='none')
    axes[0].add_patch(rect1)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Depth map 
    depth_display = axes[1].imshow(depth, cmap=custom_cmap,
                                 vmin=depth_min, vmax=depth_max)
    plt.colorbar(depth_display, ax=axes[1], 
                label=f'Depth (Range: {depth_min:.2f}-{depth_max:.2f})\nHIGHER = CLOSER')
    
    # Draw center box on depth map
    rect2 = plt.Rectangle((x_start, y_start), box_width, box_height,
                         linewidth=2, edgecolor='r', facecolor='none')
    axes[1].add_patch(rect2)
    axes[1].set_title('Depth Map (Brighter = Closer)')
    axes[1].axis('off')
    
    # Extract center region
    center_depth = depth[y_start:y_start+box_height, x_start:x_start+box_width]
    
    # For the third plot, only show the center region, aligned with the original image
    axes[2].set_xlim(0, w)
    axes[2].set_ylim(h, 0)  # Reversed y-axis to match image coordinates

    # Display only the center region in its correct position
    depth_img = axes[2].imshow(center_depth, cmap=custom_cmap, 
                            extent=[x_start, x_start + box_width, 
                                    y_start + box_height, y_start], vmin=depth_min, vmax=depth_max)
    plt.colorbar(depth_img, ax=axes[2], 
                label=f'Depth (Range: {depth_min:.2f}-{depth_max:.2f})\nHIGHER = CLOSER')

    # Draw the red box in the same position as the other plots
    rect3 = plt.Rectangle((x_start, y_start), box_width, box_height,
                        linewidth=2, edgecolor='r', facecolor='none')
    axes[2].add_patch(rect3)

    # Create danger overlay the same size as center_depth
    danger_overlay = np.zeros((center_depth.shape[0], center_depth.shape[1], 4))

    # Column analysis code remains the same
    column_width = center_depth.shape[1] // num_columns
    column_indices = [i * column_width for i in range(num_columns)]

    if column_indices[-1] + column_width <= center_depth.shape[1]:
        column_end_indices = [idx + column_width for idx in column_indices]
    else:
        column_end_indices = [idx + column_width for idx in column_indices[:-1]] + [center_depth.shape[1]]

    for i, (col_start, col_end) in enumerate(zip(column_indices, column_end_indices)):
        col = center_depth[:, col_start:col_end].mean(axis=1)
        danger, (trim_start, trim_end) = calculate_danger(col)
        
        # Convert the column coordinates to the original image coordinates
        orig_col_start = x_start + col_start
        orig_col_end = x_start + col_end
        orig_trim_start = y_start + trim_start
        orig_trim_end = y_start + trim_end
        
        # Draw rectangles on the original coordinates
        column_rect = plt.Rectangle((orig_col_start, y_start), 
                                orig_col_end - orig_col_start, 
                                box_height,
                                linewidth=1, edgecolor='white', facecolor='none')
        axes[2].add_patch(column_rect)
        
        analyzed_rect = plt.Rectangle((orig_col_start, orig_trim_start), 
                                    orig_col_end - orig_col_start, 
                                    orig_trim_end - orig_trim_start,
                                    linewidth=2, edgecolor='cyan', facecolor='none')
        axes[2].add_patch(analyzed_rect)
        
        # Fill danger overlay in local coordinates
        danger_color = np.array([1, 0, 0, danger * 0.7])
        danger_overlay[trim_start:trim_end, col_start:col_end] = \
            np.tile(danger_color, (trim_end-trim_start, col_end-col_start, 1))
        
        # Add danger value text in original coordinates
        y_pos = y_start + center_depth.shape[0] // 2
        axes[2].text(orig_col_start + (orig_col_end - orig_col_start) // 2, y_pos, 
                    f"{danger:.2f}", color='white', fontweight='bold', ha='center', 
                    bbox=dict(facecolor='black', alpha=0.5))

    # Add the danger overlay in the correct position
    axes[2].imshow(danger_overlay, extent=[x_start, x_start + box_width, 
                                        y_start + box_height, y_start], 
                alpha=0.7)
    axes[2].set_title('Depth Map with Danger Assessment')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()


image_path = "..\\clean_dataset\\dataset_rotated\\20240325-155643\\84788359.jpg"
depth_path = "..\\clean_dataset\\dataset_depth\\20240325-155643\\84788359-dpt_beit_large_512.png"
print(f"Processing {image_path} and {depth_path}")

# Analyze with 5 columns and specify a depth range
plot_simple_depth_with_danger(image_path, depth_path, num_columns=3, depth_range=(0, 1))

When we're happy, we can convert the entire dataset into an efficient H5 file for training. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import h5py
import os
import tqdm
import pandas as pd
import json
from concurrent.futures import ProcessPoolExecutor
import multiprocessing

def calculate_image_danger_values(image_path, depth_path, num_columns=5):
    """
    Calculate danger values for an image and its depth map
    
    Parameters:
    - image_path: Path to image
    - depth_path: Path to depth map
    - num_columns: Number of columns to analyze
    
    Returns:
    - danger_values: List of danger values for each column
    """
    try:
        # Load depth map
        depth = plt.imread(depth_path)
        
        # Convert depth to float32 if needed
        if depth.dtype == np.uint8:
            depth = depth.astype(np.float32) / 255.0
        
        # Get dimensions of depth map
        h, w = depth.shape[:2]
        
        # Define center box (square with height of image)
        box_width = int(h)
        box_height = int(h)
        y_start = (h - box_height) // 2
        x_start = (w - box_width) // 2
        
        # Extract center region
        center_depth = depth[y_start:y_start+box_height, x_start:x_start+box_width]
        
        # Divide the center region into evenly spaced columns
        column_width = center_depth.shape[1] // num_columns
        column_indices = [i * column_width for i in range(num_columns)]
        
        # Add the right edge if needed
        if column_indices[-1] + column_width <= center_depth.shape[1]:
            column_end_indices = [idx + column_width for idx in column_indices]
        else:
            column_end_indices = [idx + column_width for idx in column_indices[:-1]] + [center_depth.shape[1]]
        
        # Calculate danger for each column
        danger_values = []
        
        for i, (col_start, col_end) in enumerate(zip(column_indices, column_end_indices)):
            # Extract the full column and average across column width
            col = center_depth[:, col_start:col_end].mean(axis=1)
            
            # Calculate danger
            danger, _ = calculate_danger(col)
            danger_values.append(danger)
        
        return danger_values, True
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return [0.0] * num_columns, False

def process_image_pair(args):
    """Process a single image/depth pair - for parallel processing"""
    image_path, depth_path, num_columns, idx = args
    danger_values, success = calculate_image_danger_values(image_path, depth_path, num_columns)
    return {
        'image_path': str(image_path),
        'depth_path': str(depth_path),
        'danger_values': danger_values,
        'success': success,
        'index': idx
    }

def create_danger_value_dataset(base_dir, output_dir, num_columns=5, parallel=False, num_workers=None):
    """
    Process an entire dataset and create danger value labels
    
    Parameters:
    - base_dir: Base directory containing dataset_rotated and dataset_depth folders
    - output_dir: Directory to save the output files
    - num_columns: Number of columns to analyze per image
    - parallel: Whether to use parallel processing
    - num_workers: Number of worker processes (defaults to CPU count - 1)
    
    Returns:
    - None (saves output files)
    """
    base_path = Path(base_dir)
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True, parents=True)
    
    # Get all image and depth folders
    image_folders = list((base_path / 'dataset_rotated').glob('*'))
    depth_folders = list((base_path / 'dataset_depth').glob('*'))
    
    # Check that folder names match
    image_folder_names = {folder.name for folder in image_folders}
    depth_folder_names = {folder.name for folder in depth_folders}
    common_folders = image_folder_names.intersection(depth_folder_names)
    
    print(f"Found {len(common_folders)} common folders between images and depths")
    
    # Lists to store all image/depth pairs
    all_images = []
    all_depths = []
    
    # Match images with their corresponding depth maps
    for folder_name in common_folders:
        image_folder = base_path / 'dataset_rotated' / folder_name
        depth_folder = base_path / 'dataset_depth' / folder_name

        # print(f"Processing folders: {image_folder}, {depth_folder}")

        # Get all images and depths in this folder
        images = list(image_folder.glob('*.jpg')) + list(image_folder.glob('*.png'))
        depths = list(depth_folder.glob('*.jpg')) + list(depth_folder.glob('*.png'))
        
        # Match by filename (without extension)
        image_dict = {img.stem: img for img in images}
        depth_dict = {d.stem: d for d in depths}

        # Normalize depth filenames by removing '-dpt_beit_large_512'
        depth_dict_normalized = {d.stem.replace('-dpt_beit_large_512', ''): d for d in depths}

        # Find matching pairs
        common_names = set(image_dict.keys()).intersection(set(depth_dict_normalized.keys()))

        for name in common_names:
            all_images.append(image_dict[name])
            all_depths.append(depth_dict_normalized[name])

    
    print(f"Found {len(all_images)} matching image/depth pairs")
    
    # Process all pairs
    results = []
    
    if parallel:
        if num_workers is None:
            num_workers = max(1, multiprocessing.cpu_count() - 1)
        
        print(f"Processing with {num_workers} workers")
        
        # Create arguments for parallel processing
        args_list = [(all_images[i], all_depths[i], num_columns, i) 
                     for i in range(len(all_images))]
        
        # Process in parallel
        with ProcessPoolExecutor(max_workers=num_workers) as executor:
            for result in tqdm.tqdm(executor.map(process_image_pair, args_list), 
                                   total=len(args_list), 
                                   desc="Processing images"):
                if result['success']:
                    results.append(result)
    else:
        # Process sequentially
        for i, (image_path, depth_path) in enumerate(tqdm.tqdm(zip(all_images, all_depths), 
                                                              total=len(all_images),
                                                              desc="Processing images")):
            danger_values, success = calculate_image_danger_values(image_path, depth_path, num_columns)
            
            if success:
                results.append({
                    'image_path': str(image_path),
                    'depth_path': str(depth_path),
                    'danger_values': danger_values,
                    'index': i
                })
    
    print(f"Successfully processed {len(results)} image/depth pairs")
    
    # Create a DataFrame for easy analysis
    df_data = []
    for result in results:
        row = {
            'image_path': result['image_path'],
            'depth_path': result['depth_path']
        }
        
        # Add danger values as separate columns
        for i, val in enumerate(result['danger_values']):
            row[f'danger_col_{i}'] = val
        
        # Calculate key direction values (left, center, right)
        if num_columns >= 3:
            left_idx = 0
            center_idx = num_columns // 2
            right_idx = num_columns - 1
            
            row['danger_left'] = result['danger_values'][left_idx]
            row['danger_center'] = result['danger_values'][center_idx]
            row['danger_right'] = result['danger_values'][right_idx]
            
            # Find recommended direction (lowest danger)
            key_dangers = [row['danger_left'], row['danger_center'], row['danger_right']]
            directions = ["left", "center", "right"]
            best_direction = directions[np.argmin(key_dangers)]
            row['best_direction'] = best_direction
        
        df_data.append(row)
    
    # Create DataFrame
    df = pd.DataFrame(df_data)
    
    # Save results in multiple formats
    
    # 1. CSV file - good for data exploration and analysis
    csv_path = output_path / "danger_values.csv"
    df.to_csv(csv_path, index=False)
    print(f"Saved CSV to {csv_path}")
    
    # 2. HDF5 file - efficient for ML training
    h5_path = output_path / "danger_values.h5"
    with h5py.File(h5_path, 'w') as f:
        # Store image paths as a dataset
        dt = h5py.special_dtype(vlen=str)
        image_paths_ds = f.create_dataset('image_paths', (len(results),), dtype=dt)
        depth_paths_ds = f.create_dataset('depth_paths', (len(results),), dtype=dt)
        
        for i, result in enumerate(results):
            image_paths_ds[i] = result['image_path']
            depth_paths_ds[i] = result['depth_path']
        
        # Store danger values as a 2D array
        danger_values = np.array([r['danger_values'] for r in results])
        f.create_dataset('danger_values', data=danger_values)
        
        # Store additional attributes
        f.attrs['num_columns'] = num_columns
        f.attrs['creation_date'] = str(pd.Timestamp.now())
        f.attrs['total_images'] = len(results)
    
    print(f"Saved HDF5 to {h5_path}")
    
    # 3. JSON file - for easy inspection
    json_path = output_path / "danger_values.json"
    with open(json_path, 'w') as f:
        json.dump({
            'metadata': {
                'num_columns': num_columns,
                'total_images': len(results),
                'creation_date': str(pd.Timestamp.now())
            },
            'data': results
        }, f, indent=2)
    
    print(f"Saved JSON to {json_path}")
    
    return df, h5_path, csv_path, json_path

In [ ]:
base_dir = "..\\clean_dataset" 
output_dir = "..\\danger_labels"

# Process the dataset
df, h5_path, csv_path, json_path = create_danger_value_dataset(
    base_dir=base_dir,
    output_dir=output_dir,
    num_columns=3,
    parallel=False
)

Now we have useful labels in theory, but we can apply some additional preprocessing to make our training process faster by converting to yuv and rotating to the expected orientation of the drone - so we don't need to do this for every training iteration.

In [ ]:
import h5py
import cv2
import numpy as np
from tqdm import tqdm

def preprocess_and_save(original_h5_path, output_h5_path):
    with h5py.File(original_h5_path, 'r') as f_orig:
        image_paths = [x.decode() for x in f_orig['image_paths']]
        danger_values = f_orig['danger_values'][:]
        num_columns = f_orig.attrs.get('num_columns', 5)
        
        with h5py.File(output_h5_path, 'w') as f_new:
            # Create datasets
            preprocessed = f_new.create_dataset(
                'preprocessed_images',
                shape=(len(image_paths), 3, 240, 240),
                dtype=np.uint8
            )
            f_new.create_dataset('danger_values', data=danger_values)
            f_new.attrs['num_columns'] = num_columns
            
            for i, path in enumerate(tqdm(image_paths)):
                img = cv2.imread(path)
                if img is None:
                    print(f"Warning: Skipping invalid image {path}")
                    continue
                
                # Preprocessing steps
                img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
                h, w = img.shape[:2]
                y_start = (h - 240) // 2
                x_start = (w - 240) // 2
                cropped = img[y_start:y_start+240, x_start:x_start+240]
                yuv = cv2.cvtColor(cropped, cv2.COLOR_BGR2YUV)
                yuv = yuv.transpose(2, 0, 1)  # CHW format
                
                preprocessed[i] = yuv

preprocess_and_save('..\\danger_labels\\danger_values.h5', '..\\preprocessed_labels\\danger_values.h5')

